# §12.3.5 — 스펙트럼 반경 제어에 따른 학습 가능 의존 길이

> 딥러닝 교재 · 3부 12장 3절 5항 (🐍)
> 선행: §12.3.1(반복 곱) · §12.3.2(스펙트럼 조건) · §12.3.6(상대 비율의 문제)

## 이 노트북이 답하는 질문

1. **정리 12.3.1의 감쇠율이 실측과 맞는가?** 로그 축의 기울기로 확인한다.
2. **학습 가능한 최대 의존 길이는 $\rho$의 함수로 어떻게 변하는가?**
3. **학습은 $\rho$를 임계로 끌고 가는가?** 초기화별 $\rho$의 이동 궤적을 본다.

**예상 실행 시간** CPU 약 2분 (`FAST = True`이면 약 50초).

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 기울기 감쇠의 실측 — 학습 없이

무작위 $W_h$를 스펙트럼 반경 $\rho$로 맞춘 뒤 $\lVert\partial h_T/\partial h_t\rVert$를 거리별로 잰다.
정리 12.3.1: 로그 축에서 기울기 $\approx\log(\gamma\rho)$의 직선이어야 한다.

In [ ]:
M = 32; T_G = 40

def make_W(rho, rn, m=M):
    W = rn.standard_normal((m, m))/np.sqrt(m)
    W *= rho/np.abs(np.linalg.eigvals(W)).max()
    return W

def grad_decay(W, rn, n_probe=8):
    # 무작위 입력 경로에서 J_t = diag(1-tanh^2(z_t)) W 의 곱 노름
    Wx = rn.standard_normal((1, M))
    out = np.zeros((n_probe, T_G))
    for p in range(n_probe):
        h = np.zeros(M); zs = []
        xs = rn.standard_normal(T_G)
        for t in range(T_G):
            z = h @ W + xs[t]*Wx[0]
            zs.append(z); h = np.tanh(z)
        Jprod = np.eye(M)
        for k, z in enumerate(reversed(zs)):
            D = 1 - np.tanh(z)**2
            Jprod = Jprod @ (D[:, None] * W)      # (D W)^T 곱을 노름으로만 볼 것이라 순서 무관
            out[p, k] = np.linalg.norm(Jprod, 2)
    return out.mean(axis=0)

RHOS = [0.5, 0.8, 0.95, 1.05, 1.3, 2.0, 3.0]
decays = {}
rn0 = np.random.default_rng(SEED)
for rho in RHOS:
    decays[rho] = grad_decay(make_W(rho, rn0), rn0)
    print(f"rho={rho}:  ||J^(10)||={decays[rho][9]:.2e}  ||J^(30)||={decays[rho][29]:.2e}")

---
## 2. 지연 회상 과제 — $\rho$와 $\Delta$를 훑는다

$y=\operatorname{sign}(x_{T-\Delta})$: 입력 열에서 $\Delta$스텝 전 값의 부호를 마지막 상태에서 판독한다.
절단 없는 완전한 BPTT로 학습하므로, 실패의 원인은 오직 소실/폭발이다.

In [ ]:
def train_recall(rho, delta, steps=None, seed=0, track_rho=False):
    steps = steps or (120 if FAST else 250)
    T_ = delta + 5
    rn = np.random.default_rng(seed)
    Wh = make_W(rho, rn); Wx = rn.standard_normal((1, M))*1.0; b = np.zeros(M)
    U = rn.standard_normal(M)/np.sqrt(M); c = np.zeros(1)
    params = [Wh, Wx, b, U, c]
    ms = [np.zeros_like(p) for p in params]; vs = [np.zeros_like(p) for p in params]
    rb = np.random.default_rng(900+seed); B = 96
    rhos = []
    for t in range(1, steps+1):
        X = rb.standard_normal((B, T_))
        y01 = (X[:, T_-1-delta] > 0).astype(float)
        H = np.zeros((B, T_+1, M)); Z = np.zeros((B, T_, M))
        for tt in range(T_):
            Z[:, tt] = H[:, tt] @ Wh + X[:, tt:tt+1] @ Wx + b
            H[:, tt+1] = np.tanh(Z[:, tt])
        logit = H[:, T_] @ U + c
        p = np.where(logit >= 0, 1/(1+np.exp(-logit)), np.exp(logit)/(1+np.exp(logit))).ravel()
        dlogit = (p - y01)/B
        gU = H[:, T_].T @ dlogit; gc = np.array([dlogit.sum()])
        delta_b = np.outer(dlogit, U)
        gWh = np.zeros_like(Wh); gWx = np.zeros_like(Wx); gb = np.zeros_like(b)
        for tt in range(T_-1, -1, -1):
            dz = delta_b * (1 - np.tanh(Z[:, tt])**2)
            gWh += H[:, tt].T @ dz; gWx += X[:, tt:tt+1].T @ dz; gb += dz.sum(0)
            delta_b = dz @ Wh.T
        gn = np.sqrt(sum((g**2).sum() for g in [gWh, gWx, gb, gU, gc]))
        if gn > 5.0:                                  # 폭발 대비 최소한의 클리핑 (§12.5)
            sc = 5.0/gn
            gWh *= sc; gWx *= sc; gb *= sc; gU *= sc; gc *= sc
        for pp, g, m_, v_ in zip(params, [gWh, gWx, gb, gU, gc], ms, vs):
            m_[:] = 0.9*m_ + 0.1*g; v_[:] = 0.999*v_ + 0.001*g*g
            pp -= 4e-3*(m_/(1-0.9**t))/(np.sqrt(v_/(1-0.999**t))+1e-8)
        if track_rho and (t % 10 == 0 or t == 1):
            rhos.append(np.abs(np.linalg.eigvals(Wh)).max())
    # 평가
    Xev = np.random.default_rng(SEED+1).standard_normal((1500, T_))
    yev = (Xev[:, T_-1-delta] > 0)
    h = np.zeros((1500, M))
    for tt in range(T_):
        h = np.tanh(h @ Wh + Xev[:, tt:tt+1] @ Wx + b)
    acc = np.mean(((h @ U + c) > 0) == yev)
    return acc, rhos

DELTAS = [2, 10] if FAST else [2, 5, 10, 15, 20]
acc_map = np.zeros((len(RHOS), len(DELTAS)))
for ri, rho in enumerate(RHOS):
    for di, d in enumerate(DELTAS):
        acc_map[ri, di], _ = train_recall(rho, d, seed=1)
    print(f"rho={rho}: acc={np.round(acc_map[ri], 2)}  ({time.time()-_t0:.0f}초)")

# 최대 학습 가능 길이 (acc>0.9 기준)
max_len = []
for ri in range(len(RHOS)):
    ok = [DELTAS[di] for di in range(len(DELTAS)) if acc_map[ri, di] > 0.9]
    max_len.append(max(ok) if ok else 0)

# (d) rho 궤적
trajs = {}
for rho0 in ([0.5, 0.95] if FAST else [0.5, 0.8, 0.95, 1.2]):
    _, tr = train_recall(rho0, 10, seed=3, track_rho=True)
    trajs[rho0] = tr
print("궤적 수집 완료")

---
## 3. 교재 그림 — fig_12_3_5

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.9))
axes = axes.ravel()

# (a) 감쇠 곡선
ax = axes[0]
ks = np.arange(1, T_G+1)
for i, rho in enumerate(RHOS):
    ax.semilogy(ks, decays[rho], '-', color=CB[i+1], lw=1.4, label=f'$\\rho={rho}$')
ax.set_xlabel(lab('시간 거리 $k$', 'distance $k$'))
ax.set_ylabel(lab('$\\|\\partial h_T/\\partial h_{T-k}\\|_2$', 'gradient norm'))
ax.set_title(lab('(a) 감쇠는 로그 축의 직선 — 정리 12.3.1', '(a) exponential decay'), fontsize=10)
ax.legend(fontsize=8)

# (b) rho-Delta 성공 지도
ax = axes[1]
imv = ax.imshow(acc_map, cmap='viridis', vmin=0.5, vmax=1.0, aspect='auto'); ax.grid(False)
plt.colorbar(imv, ax=ax, fraction=0.046)
ax.set_xticks(range(len(DELTAS))); ax.set_xticklabels(DELTAS)
ax.set_yticks(range(len(RHOS))); ax.set_yticklabels(RHOS)
ax.set_xlabel(lab('의존 거리 $\\Delta$', 'dependency $\\Delta$'))
ax.set_ylabel(lab('초기 스펙트럼 반경 $\\rho$', 'initial $\\rho$'))
ax.set_title(lab('(b) $\\rho$--$\\Delta$ 평면의 시험 정확도', '(b) accuracy map'), fontsize=10)

# (c) 최대 학습 가능 길이
ax = axes[2]
ax.plot(RHOS, max_len, 'o-', color=CB[5], ms=6)
ax.axvline(1.0, color=CB[4], lw=0.8, ls=':')
ax.text(1.01, max(max_len)*0.55, lab('임계 $\\rho=1$', 'critical'), color=CB[4], fontsize=9)
ax.set_xlabel(lab('초기 스펙트럼 반경 $\\rho$', 'initial $\\rho$'))
ax.set_ylabel(lab('학습 가능 최대 $\\Delta$ (정확도 0.9 기준)', 'max learnable $\\Delta$'))
ax.set_title(lab('(c) 임계 근방에서만 장거리가 산다', '(c) longest learnable dependency'), fontsize=10)

# (d) 학습 중 rho의 이동
ax = axes[3]
for i, (rho0, tr) in enumerate(trajs.items()):
    ax.plot(np.arange(len(tr))*10, tr, '-', color=CB[i+1], lw=1.4, label=f'$\\rho_0={rho0}$')
ax.axhline(1.0, color='k', lw=0.7, ls=':')
ax.set_xlabel(lab('학습 걸음', 'step'))
ax.set_ylabel(lab('$\\rho(W_h)$', '$\\rho(W_h)$'))
ax.set_title(lab('(d) 학습이 $\\rho$를 임계 쪽으로 끈다 ($\\Delta=10$)', '(d) drift toward criticality'), fontsize=10)
ax.legend(fontsize=8)

save_book_fig(fig, 'fig_12_3_5')
plt.show()

> ### 읽는 법
>
> (a) 감쇠는 로그 축의 직선이고 순서는 $\rho$를 따른다. 주목할 것: $\rho>1$ 곡선조차 (느리게) 감쇠한다.
> $\tanh$ 포화가 $D_t$를 깎아 **유효 이득** $\gamma_{\text{eff}}\rho$를 1 아래로 끌어내리기 때문 —
> 폭발 조건이 충분조건이 아니라던 정리 12.3.1(ii)의 실물이다.
> (b, c) 학습 가능한 의존 길이의 정점은 원시 $\rho=1$이 아니라 **조금 위**(여기서는 1.3 근처)에 있다.
> 포화가 유효 이득을 깎는 만큼 원시 반경이 보상해야 하기 때문이다. 훨씬 키우면(3.0) 포화가 심해져
> $D_t$가 죽고 성능이 무너진다. 장거리 신호가 사는 곳은 원시 $\rho$가 아니라
> **유효 이득 $\approx1$의 좁은 띠**다 (§12.3.2).
> (d) 과제가 장거리를 요구하면 학습이 $\rho$를 그 띠 쪽으로 끌고 간다. 멀리서 출발할수록
> 끌고 갈 신호 자체가 약해 느리다 — 초기화가 중요한 이유(7장)의 순환판.

---
## 4. 자기 점검

1. (a)에서 감쇠 직선의 기울기를 읽어 $\log(\gamma\rho)$와 비교하라. $\gamma$를 얼마로 봐야 잘 맞는가? 왜 1보다 작은가?
2. (b)에서 $\rho=1.3$이 $\rho=0.5$보다 오히려 나은 칸이 있는가? 있다면 어떤 기제인가? (힌트: 포화가 사실상의 게이트 노릇을 한다)
3. 이 실험은 최소한의 클리핑(임계 5.0)을 켰다. 껐을 때 $\rho=1.3$ 행이 어떻게 변하는지 확인하라 (§12.5).
4. (d)에서 $\rho_0=1.2$의 궤적은 어디로 가는가? 아래에서 접근할 때와 무엇이 다른가?

## 5. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `RHOS`, `DELTAS` | 1–2절 | — | 지도의 해상도 |
| 클리핑 임계 | 2절 | 5.0 | 폭발 체제의 운명이 갈린다 |
| `M` | 1절 | 32 | 상태 크기 |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")